# Project: Multi-Agent Research System

This project combines every pattern from the multi-agent section into a working research pipeline:
- **Supervisor** — plans targeted search queries
- **Parallel execution via Send API** — fan-out to N search agents dynamically
- **Blackboard** — shared `findings` field accumulates results from all agents
- **Iterative quality loop** — quality checker loops back to report writer until score ≥ 0.7

```
START
  │
  ▼
supervisor  ─── plans 3 queries ──────────────────────────────────────┐
  │                                                                    │
  ├──► search_agent [query 1]  ─┐                                     │
  ├──► search_agent [query 2]  ─┼──► analyst ──► report_writer ──► quality_checker
  └──► search_agent [query 3]  ─┘                      ▲               │
       (Send API fan-out)                               └───────────────┘
                                                   (if score < 0.7, revise)
                                                           │
                                                          END
```

### The Send API

```python
from langgraph.types import Send

def dispatch_searches(state: ResearchState) -> list[Send]:
    return [
        Send("search_agent", {"search_query": query, "findings": []})
        for query in state["search_queries"]
    ]
```

`Send(node_name, state)` creates a **dynamic task** — one invocation of `search_agent` per query, all running in parallel. The number of tasks isn't known at build time, which is what makes this different from static fan-out.

In [ ]:
import json, operator
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Send
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from typing_extensions import TypedDict, Annotated
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
creative_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

## 1. State Schema

Two state types serve different purposes:

| State | Used by | Key fields |
|-------|---------|------------|
| `ResearchState` | Parent graph | `findings: Annotated[list[dict], operator.add]` — accumulates across all search agents |
| `SearchTaskState` | Individual search agent (via Send) | `search_query: str`, `findings` — isolated per task |

The `operator.add` reducer on `findings` is what makes fan-in work: each `search_agent` appends to `findings`, and LangGraph merges them all into the parent state automatically.

In [ ]:
class ResearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    topic: str
    search_queries: list[str]
    findings: Annotated[list[dict], operator.add]  # accumulates from all search agents
    analysis: str
    report: str
    quality_score: float
    quality_feedback: str
    iteration: int


# State for individual search tasks dispatched via Send API
class SearchTaskState(TypedDict):
    search_query: str
    findings: Annotated[list[dict], operator.add]

## 2. Node Definitions

In [ ]:
# ------------------------------------------------------------------
# Supervisor: plans the research by generating targeted queries
# ------------------------------------------------------------------
def supervisor(state: ResearchState) -> dict:
    response = llm.invoke([
        SystemMessage(content=(
            "You are a research supervisor. Given a topic, generate exactly 3 "
            "specific search queries covering different angles. "
            "Return ONLY a JSON array of strings. No markdown."
        )),
        HumanMessage(content=f"Research topic: {state['topic']}"),
    ])
    try:
        queries = json.loads(response.content)
    except json.JSONDecodeError:
        queries = [
            f"{state['topic']} overview",
            f"{state['topic']} latest developments",
            f"{state['topic']} practical applications",
        ]
    return {
        "search_queries": queries[:3],
        "messages": [AIMessage(content=f"[SUPERVISOR]: Planned queries: {queries}", name="supervisor")],
    }


# ------------------------------------------------------------------
# Search Agent: executes one query — launched in parallel via Send
# ------------------------------------------------------------------
def search_agent(state: SearchTaskState) -> dict:
    query = state["search_query"]
    response = llm.invoke([
        SystemMessage(content=(
            "You are a web research agent. For the given search query, provide 2-3 key findings. "
            "Each finding should have a 'title' and 'detail' field. Return a JSON array. No markdown."
        )),
        HumanMessage(content=f"Search query: {query}"),
    ])
    try:
        results = json.loads(response.content)
    except json.JSONDecodeError:
        results = [{"title": query, "detail": response.content}]

    for r in results:
        r["source_query"] = query  # tag for provenance

    return {"findings": results}  # operator.add merges this into ResearchState.findings


# ------------------------------------------------------------------
# Send API fan-out: returns one Send per query
# ------------------------------------------------------------------
def dispatch_searches(state: ResearchState) -> list[Send]:
    """Dynamically dispatch one search_agent per query — all run in parallel."""
    return [
        Send("search_agent", {"search_query": query, "findings": []})
        for query in state["search_queries"]
    ]


# ------------------------------------------------------------------
# Analyst: synthesizes all findings into a structured analysis
# ------------------------------------------------------------------
def analyst(state: ResearchState) -> dict:
    findings_text = json.dumps(state["findings"], indent=2)
    response = llm.invoke([
        SystemMessage(content=(
            "You are a research analyst. Synthesize the collected findings. Identify:\n"
            "1. Key themes across all findings\n"
            "2. Any contradictions or gaps\n"
            "3. The most important insights\n\n"
            "Write 2-3 paragraphs."
        )),
        HumanMessage(content=f"Research topic: {state['topic']}\n\nFindings:\n{findings_text}"),
    ])
    return {
        "analysis": response.content,
        "messages": [AIMessage(content=f"[ANALYST]: {response.content}", name="analyst")],
    }


# ------------------------------------------------------------------
# Report Writer: produces a structured Markdown report
# Includes quality feedback in its prompt on revision rounds
# ------------------------------------------------------------------
def report_writer(state: ResearchState) -> dict:
    revision_note = ""
    if state["iteration"] > 0 and state.get("quality_feedback"):
        revision_note = (
            f"\n\nIMPORTANT — Revision #{state['iteration']}. "
            f"Address this feedback: {state['quality_feedback']}"
        )

    response = creative_llm.invoke([
        SystemMessage(content=(
            "You are a report writer. Produce a well-structured research report with:\n"
            "1. Executive Summary (2-3 sentences)\n"
            "2. Key Findings (bullet points)\n"
            "3. Analysis (1-2 paragraphs)\n"
            "4. Recommendations (3 actionable items)\n\n"
            "Use markdown formatting."
            f"{revision_note}"
        )),
        HumanMessage(content=(
            f"Topic: {state['topic']}\n\n"
            f"Analysis:\n{state['analysis']}\n\n"
            f"Raw findings:\n{json.dumps(state['findings'][:6], indent=2)}"
        )),
    ])
    return {
        "report": response.content,
        "messages": [AIMessage(
            content=f"[REPORT WRITER]: Report {'revised' if state['iteration'] > 0 else 'drafted'}.",
            name="report_writer",
        )],
    }


# ------------------------------------------------------------------
# Quality Checker: scores and approves or rejects the report
# ------------------------------------------------------------------
class QualityReview(BaseModel):
    score: float = Field(description="Quality score from 0.0 to 1.0")
    feedback: str = Field(description="Specific feedback for improvement")
    approved: bool = Field(description="Whether the report meets quality standards")


def quality_checker(state: ResearchState) -> dict:
    review_llm = llm.with_structured_output(QualityReview)
    review = review_llm.invoke([
        SystemMessage(content=(
            "You are a quality reviewer. Score the report on completeness, clarity, and actionability. "
            "Score from 0.0 to 1.0. Approve if score >= 0.7. "
            "If this is iteration 2 or higher, be more lenient."
        )),
        HumanMessage(content=(
            f"Topic: {state['topic']}\n"
            f"Iteration: {state['iteration']}\n\n"
            f"Report:\n{state['report']}"
        )),
    ])

    # Force approve after 2 iterations to prevent infinite loops
    approved = review.approved or state["iteration"] >= 2

    return {
        "quality_score": review.score,
        "quality_feedback": review.feedback,
        "iteration": state["iteration"] + 1,
        "messages": [AIMessage(
            content=(
                f"[QUALITY CHECK]: Score {review.score:.1f} — "
                f"{'APPROVED' if approved else 'REVISION NEEDED'}: {review.feedback}"
            ),
            name="quality_checker",
        )],
    }


def quality_gate(state: ResearchState) -> Literal["report_writer", "end"]:
    if state["quality_score"] >= 0.7 or state["iteration"] >= 2:
        return "end"
    return "report_writer"

## 3. Assemble the Graph

The key wiring is the `dispatch_searches` conditional edge from `supervisor` — it returns `list[Send]` instead of a string, which tells LangGraph to create one parallel task per item.

In [ ]:
def create_research_system():
    g = StateGraph(ResearchState)

    g.add_node("supervisor",      supervisor)
    g.add_node("search_agent",    search_agent)
    g.add_node("analyst",         analyst)
    g.add_node("report_writer",   report_writer)
    g.add_node("quality_checker", quality_checker)

    g.add_edge(START, "supervisor")

    # Dynamic fan-out: returns list[Send] — one search_agent per query
    g.add_conditional_edges("supervisor", dispatch_searches, ["search_agent"])

    # All search agents fan-in to analyst
    g.add_edge("search_agent",    "analyst")
    g.add_edge("analyst",         "report_writer")
    g.add_edge("report_writer",   "quality_checker")

    # Quality gate: approve or loop back to revise
    g.add_conditional_edges(
        "quality_checker", quality_gate,
        {"report_writer": "report_writer", "end": END},
    )

    return g.compile()


research_system = create_research_system()
print(research_system.get_graph().draw_mermaid())

## 4. Stream Execution

`system.stream(state, stream_mode="updates")` yields one dict per node completion. Each dict maps `node_name → state_update`, letting you observe progress in real time.

In [ ]:
topic = "Best practices for building multi-agent AI systems"
print(f"Research topic: {topic}\n")

initial_state = {
    "messages": [],
    "topic": topic,
    "search_queries": [],
    "findings": [],
    "analysis": "",
    "report": "",
    "quality_score": 0.0,
    "quality_feedback": "",
    "iteration": 0,
}

for step in research_system.stream(initial_state, stream_mode="updates"):
    for node_name, update in step.items():
        print(f"[{node_name}] completed")
        if "search_queries" in update and update["search_queries"]:
            print(f"  Planned queries: {update['search_queries']}")
        if "findings" in update and update["findings"]:
            print(f"  Found {len(update['findings'])} results")
        if "quality_score" in update:
            print(f"  Quality score: {update['quality_score']:.1f}")
        if "report" in update and update["report"]:
            print(f"  Report length: {len(update['report'])} chars")
    print()

## 5. Full Run — Final Report

In [ ]:
topic = "The impact of AI agents on software development in 2026"
print(f"Topic: {topic}\n")

result = research_system.invoke({
    "messages": [],
    "topic": topic,
    "search_queries": [],
    "findings": [],
    "analysis": "",
    "report": "",
    "quality_score": 0.0,
    "quality_feedback": "",
    "iteration": 0,
})

print("Agent Activity Log:")
print("-" * 40)
for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        print(f"  {msg.content[:120]}...")

print(f"\nTotal findings: {len(result['findings'])}")
print(f"Quality score:  {result['quality_score']:.1f}")
print(f"Iterations:     {result['iteration']}")
print("\n" + "=" * 60)
print("FINAL RESEARCH REPORT")
print("=" * 60)
print(result["report"])